# Import Data from AWS S3 to GCS with Dataplex Custom Lineage

This notebook demonstrates cross-cloud lineage tracking by importing American Community Survey (ACS) data from AWS S3 into Google Cloud Platform, then establishing the complete data provenance chain in Dataplex Universal Catalog.

## What This Notebook Does

1. **Connects to Public AWS S3** - Accesses the Data.world ACS dataset (public bucket)
2. **Transfers Data to GCS** - Downloads CSV from S3 and uploads to Google Cloud Storage
3. **Loads into BigQuery** - Creates a BigQuery table from the GCS CSV
4. **Creates Custom Dataplex Entries** - Registers the S3 source in Dataplex Universal Catalog
5. **Establishes Lineage** - Uses the Data Lineage API to track the S3 → GCS → BigQuery journey

## Cross-Cloud Lineage Architecture

```
AWS S3 (External)                    Google Cloud Platform
┌─────────────────┐                 ┌──────────────────────┐
│ dataworld-      │   boto3         │  GCS Bucket          │
│ linked-acs      │ ─────────────>  │  census-s3-import    │
│                 │   transfer      │                      │
└─────────────────┘                 └──────────┬───────────┘
         │                                     │
         │                                     │ BigQuery load
         │ Custom Lineage Event                │
         │ (Data Lineage API)                  ▼
         │                          ┌──────────────────────┐
         └─────────────────────────>│  BigQuery Table      │
           Dataplex Catalog          │  s3_acs_data         │
                                     └──────────────────────┘
```

## Why Custom Lineage?

Dataplex doesn't automatically detect data movement from external cloud providers. By using the **Data Lineage API**, we can:

- **Track Cross-Cloud Data Movement** - Show the origin of data coming from AWS
- **Enable Impact Analysis** - Understand downstream dependencies
- **Support Compliance** - Document data provenance for auditing
- **Improve Discovery** - Search for data by its original source

## Dataset Information

**Source**: American Community Survey (ACS) public dataset on Data.world  
**Documentation**: https://registry.opendata.aws/census-dataworld-pums/  
**Tutorial**: https://docs.data.world/uscensus/#53---copying-the-linked-acs-data-from-s3-to-ec2

- **AWS S3 Bucket**: `s3://dataworld-linked-acs/`
- **Region**: us-east-1 (public access, no credentials required)
- **License**: Creative Commons Attribution 4.0 International (CC BY 4.0)

**Bucket Structure:**
- **RawData/**: ~50 compressed RDF/Turtle files (.ttl.gz) - raw census microdata
- **EnhancedData/**: ~50 compressed RDF/Turtle files (.ttl.gz) - enhanced with ontologies
- **Ontology/**: Ontology definitions for the dataset
- **TabulationQueries/**: 2 CSV summary files + 32 SPARQL query templates (.rq)
  - `pums_estimates_14.csv` (~90 KB) - 2014 estimates
  - `pums_estimates_15.csv` (~90 KB) - 2015 estimates

**This Demo**: Uses the CSV summary files to demonstrate cross-cloud lineage without requiring RDF processing or decompression.

**Production Use**: The full dataset requires:
1. Download .ttl.gz files from EnhancedData/
2. Decompress with: `gzip -d file.ttl.gz` or `tar xzvf *.ttl.gz`
3. Convert RDF/Turtle to CSV/Parquet for BigQuery
4. Or load into an RDF triplestore (e.g., Blazegraph)

## Prerequisites

- GCP project with billing enabled
- Required IAM roles:
  - `roles/storage.admin` (Cloud Storage bucket management)
  - `roles/bigquery.admin` (BigQuery dataset/table management)
  - `roles/dataplex.catalogAdmin` (Custom catalog entry management)
  - `roles/datacatalog.admin` (Data Lineage API access)

## Time Estimate

**30-40 minutes** (including file transfer and all lineage setup)

## Resources Created

1. GCS Bucket: `{project-id}-census-s3-import`
2. BigQuery Table: `census_bureau_acs.s3_acs_data`
3. Dataplex Entry Group: `aws-storage-entries`
4. Dataplex Entry Type: `s3-object`
5. Custom S3 Entry with FQN: `s3://dataworld-linked-acs/TabulationQueries/{filename}`
6. Data Lineage Process: `s3-to-gcs-import-process`
7. Data Lineage Run (per execution)
8. Lineage Events linking S3 → GCS → BigQuery

# Section 1: Configuration & Authentication

Configure project settings and verify authentication.

In [47]:
# Configuration variables

PROJECT_ID = "johnswain-1200-20260303091357"  
REGION = "us-central1"                           
CATALOG_LOCATION = "us"  

# GCS Configuration
BUCKET_NAME = f"{PROJECT_ID}-census-s3-import"

# BigQuery Configuration
BQ_DATASET_ID = "census_bureau_acs"
BQ_TABLE_ID = "s3_acs_data"

# AWS S3 Configuration (Public Bucket - No Credentials)
S3_BUCKET = "dataworld-linked-acs"
S3_PREFIX = "TabulationQueries/"
S3_REGION = "us-east-1"

# Dataplex Configuration
ENTRY_GROUP_ID = "aws-storage-entries"
ENTRY_TYPE_ID = "s3-object"
LINEAGE_PROCESS_ID = "s3-to-gcs-import-process"

print("📋 Configuration:")
print(f"  Project ID: {PROJECT_ID}")
print(f"  Region: {REGION}")
print(f"  GCS Bucket: {BUCKET_NAME}")
print(f"  BigQuery: {BQ_DATASET_ID}.{BQ_TABLE_ID}")
print(f"  S3 Source: s3://{S3_BUCKET}/{S3_PREFIX}")

📋 Configuration:
  Project ID: johnswain-1200-20260303091357
  Region: us-central1
  GCS Bucket: johnswain-1200-20260303091357-census-s3-import
  BigQuery: census_bureau_acs.s3_acs_data
  S3 Source: s3://dataworld-linked-acs/TabulationQueries/


In [48]:
# Verify authentication
from google.auth import default
import google.auth

try:
    credentials, project = default()
    print("🔐 Authentication Status:")
    print(f"  ✅ Authenticated successfully")
    print(f"  ✅ Default project: {project}")
    
    if project != PROJECT_ID:
        print(f"  ⚠️  Note: Using PROJECT_ID ({PROJECT_ID}) instead of default ({project})")
    
except Exception as e:
    print(f"❌ Authentication failed: {e}")
    print("Please run: gcloud auth application-default login")
    raise

🔐 Authentication Status:
  ✅ Authenticated successfully
  ✅ Default project: johnswain-1200-20260303091357


In [49]:
# Check required IAM permissions
print("🔐 Required IAM permissions:")
print()
print("   1️⃣  Cloud Storage:")
print("      - roles/storage.admin")
print()
print("   2️⃣  BigQuery:")
print("      - roles/bigquery.admin")
print()
print("   3️⃣  Dataplex:")
print("      - roles/dataplex.catalogAdmin")
print()
print("   4️⃣  Data Catalog:")
print("      - roles/datacatalog.admin (for Data Lineage API)")
print()

try:
    user_email = None
    if hasattr(credentials, '_service_account_email'):
        user_email = credentials._service_account_email
    elif hasattr(credentials, 'service_account_email'):
        user_email = credentials.service_account_email
    
    if user_email:
        print(f"   Your account: {user_email}")
    else:
        print("   Your account: (Unable to detect email from credentials)")
except:
    print("   Your account: (Unable to detect email)")

print()
print("💡 To grant permissions (if needed):")
print()
print(f"   gcloud projects add-iam-policy-binding {PROJECT_ID} \\")
print(f"     --member='user:YOUR_EMAIL' \\")
print(f"     --role='roles/storage.admin'")
print()
print(f"   gcloud projects add-iam-policy-binding {PROJECT_ID} \\")
print(f"     --member='user:YOUR_EMAIL' \\")
print(f"     --role='roles/bigquery.admin'")
print()
print(f"   gcloud projects add-iam-policy-binding {PROJECT_ID} \\")
print(f"     --member='user:YOUR_EMAIL' \\")
print(f"     --role='roles/dataplex.catalogAdmin'")
print()
print(f"   gcloud projects add-iam-policy-binding {PROJECT_ID} \\")
print(f"     --member='user:YOUR_EMAIL' \\")
print(f"     --role='roles/datacatalog.admin'")

🔐 Required IAM permissions:

   1️⃣  Cloud Storage:
      - roles/storage.admin

   2️⃣  BigQuery:
      - roles/bigquery.admin

   3️⃣  Dataplex:
      - roles/dataplex.catalogAdmin

   4️⃣  Data Catalog:
      - roles/datacatalog.admin (for Data Lineage API)

   Your account: (Unable to detect email from credentials)

💡 To grant permissions (if needed):

   gcloud projects add-iam-policy-binding johnswain-1200-20260303091357 \
     --member='user:YOUR_EMAIL' \
     --role='roles/storage.admin'

   gcloud projects add-iam-policy-binding johnswain-1200-20260303091357 \
     --member='user:YOUR_EMAIL' \
     --role='roles/bigquery.admin'

   gcloud projects add-iam-policy-binding johnswain-1200-20260303091357 \
     --member='user:YOUR_EMAIL' \
     --role='roles/dataplex.catalogAdmin'

   gcloud projects add-iam-policy-binding johnswain-1200-20260303091357 \
     --member='user:YOUR_EMAIL' \
     --role='roles/datacatalog.admin'


# Section 2: Install Required Libraries

Install Python packages for AWS S3, GCS, BigQuery, and Data Lineage.

In [50]:
# Install required libraries
import sys
import subprocess

print("📦 Installing required libraries...")
print()

packages = [
    "boto3",  # AWS S3 access
    "google-cloud-storage",  # GCS operations
    "google-cloud-bigquery",  # BigQuery loading
    "google-cloud-dataplex",  # Dataplex catalog
    "google-cloud-datacatalog-lineage"  # Data Lineage API
]

for package in packages:
    print(f"   Installing {package}...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

print()
print("✅ All libraries installed successfully")

📦 Installing required libraries...

   Installing boto3...



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


   Installing google-cloud-storage...



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


   Installing google-cloud-bigquery...



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


   Installing google-cloud-dataplex...



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


   Installing google-cloud-datacatalog-lineage...

✅ All libraries installed successfully



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [51]:
# Import libraries
import boto3
from botocore import UNSIGNED
from botocore.config import Config
from google.cloud import storage, bigquery, dataplex_v1
from google.cloud.datacatalog_lineage_v1 import LineageClient
from google.cloud.datacatalog_lineage_v1.types import (
    Process, Run, LineageEvent, EventLink, EntityReference,
    CreateProcessRequest, CreateRunRequest, CreateLineageEventRequest
)
from google.protobuf.timestamp_pb2 import Timestamp
from google.protobuf import struct_pb2
import io
import datetime
import time
import re

print("✅ Libraries imported successfully")

✅ Libraries imported successfully


# Section 3: Connect to AWS S3 and List Files

Configure anonymous access to the public S3 bucket and list available CSV files.

In [52]:
# Configure S3 client for anonymous access to public bucket
print("🔗 Connecting to AWS S3 public bucket...")
print()

s3_client = boto3.client(
    's3',
    region_name=S3_REGION,
    config=Config(signature_version=UNSIGNED)
)

print(f"✅ S3 client configured")
print(f"   Bucket: s3://{S3_BUCKET}/{S3_PREFIX}")
print(f"   Region: {S3_REGION}")
print(f"   Access: Anonymous (public bucket)")

🔗 Connecting to AWS S3 public bucket...

✅ S3 client configured
   Bucket: s3://dataworld-linked-acs/TabulationQueries/
   Region: us-east-1
   Access: Anonymous (public bucket)


In [53]:
# List files in S3 bucket
print(f"📂 Listing files in s3://{S3_BUCKET}/{S3_PREFIX}...")
print()

try:
    response = s3_client.list_objects_v2(
        Bucket=S3_BUCKET,
        Prefix=S3_PREFIX,
        MaxKeys=100
    )
    
    csv_files = []
    all_files = []
    
    if 'Contents' in response:
        for obj in response['Contents']:
            key = obj['Key']
            
            # Skip the prefix itself if it appears as a key
            if key == S3_PREFIX or key.endswith('/'):
                continue
            
            size_mb = obj['Size'] / (1024 * 1024)
            file_info = {
                'key': key,
                'size': obj['Size'],
                'size_mb': size_mb,
                'modified': obj['LastModified']
            }
            all_files.append(file_info)
            
            if key.endswith('.csv'):
                csv_files.append(file_info)
        
        print(f"✅ Found {len(all_files)} total files")
        print(f"✅ Found {len(csv_files)} CSV files")
        print()
        
        if csv_files:
            print("Available CSV files:")
            for i, file in enumerate(csv_files, 1):
                print(f"   {i}. {file['key']}")
                print(f"      Size: {file['size_mb']:.2f} MB ({file['size']:,} bytes)")
                print(f"      Modified: {file['modified']}")
        else:
            print("⚠️  No CSV files found in this prefix")
            print()
            print("Sample of other files found:")
            for i, file in enumerate(all_files[:5], 1):
                print(f"   {i}. {file['key']}")
                print(f"      Size: {file['size_mb']:.4f} MB")
    else:
        print("⚠️  No objects found in bucket")
        csv_files = []

except Exception as e:
    print(f"❌ Error listing S3 files: {e}")
    raise

📂 Listing files in s3://dataworld-linked-acs/TabulationQueries/...

✅ Found 36 total files
✅ Found 2 CSV files

Available CSV files:
   1. TabulationQueries/pums_estimates_14.csv
      Size: 0.09 MB (89,911 bytes)
      Modified: 2017-03-10 17:45:56+00:00
   2. TabulationQueries/pums_estimates_15.csv
      Size: 0.09 MB (89,937 bytes)
      Modified: 2017-03-10 17:45:56+00:00


In [54]:
# Select a file to transfer
if csv_files:
    selected_file = csv_files[0]
    S3_FILE_KEY = selected_file['key']
    S3_FILE_NAME = S3_FILE_KEY.split('/')[-1]
    S3_FILE_SIZE_MB = selected_file['size_mb']
    
    print(f"📌 Selected file for transfer:")
    print(f"   Key: {S3_FILE_KEY}")
    print(f"   Name: {S3_FILE_NAME}")
    print(f"   Size: {S3_FILE_SIZE_MB:.2f} MB")
    print()
    print(f"   Full S3 URI: s3://{S3_BUCKET}/{S3_FILE_KEY}")
else:
    raise ValueError("No CSV files found to transfer")

📌 Selected file for transfer:
   Key: TabulationQueries/pums_estimates_14.csv
   Name: pums_estimates_14.csv
   Size: 0.09 MB

   Full S3 URI: s3://dataworld-linked-acs/TabulationQueries/pums_estimates_14.csv


# Section 4: Create GCS Bucket

Create a Cloud Storage bucket to receive the data from S3.

In [55]:
# Initialize GCS client
storage_client = storage.Client(project=PROJECT_ID)

print("✅ Storage client initialized")

✅ Storage client initialized


In [56]:
# Create GCS bucket
print(f"🪣 Creating GCS bucket: {BUCKET_NAME}")
print()

try:
    bucket = storage_client.bucket(BUCKET_NAME)
    
    if bucket.exists():
        print(f"⚠️  Bucket '{BUCKET_NAME}' already exists")
        print(f"   Will use existing bucket")
    else:
        bucket = storage_client.create_bucket(
            BUCKET_NAME,
            location="US"  # Multi-region for BigQuery compatibility
        )
        
        # Add labels for tracking
        bucket.labels = {
            'source': 'aws-s3',
            'demo': 'dataplex-lineage',
            'dataset': 'census-acs'
        }
        bucket.patch()
        
        print(f"✅ Bucket created successfully")
        print(f"   Name: {bucket.name}")
        print(f"   Location: {bucket.location}")
        print(f"   Storage Class: {bucket.storage_class}")
    
    print()
    print(f"🔗 View bucket in console:")
    print(f"   https://console.cloud.google.com/storage/browser/{BUCKET_NAME}?project={PROJECT_ID}")

except Exception as e:
    print(f"❌ Error creating bucket: {e}")
    raise

🪣 Creating GCS bucket: johnswain-1200-20260303091357-census-s3-import

⚠️  Bucket 'johnswain-1200-20260303091357-census-s3-import' already exists
   Will use existing bucket

🔗 View bucket in console:
   https://console.cloud.google.com/storage/browser/johnswain-1200-20260303091357-census-s3-import?project=johnswain-1200-20260303091357


# Section 5: Transfer Data from S3 to GCS

Download the file from S3 and upload it to GCS using a streaming approach.

In [57]:
# Transfer file from S3 to GCS
print(f"📦 Transferring file from S3 to GCS...")
print()
print(f"   Source: s3://{S3_BUCKET}/{S3_FILE_KEY}")

# Define GCS destination
GCS_BLOB_NAME = f"data/{S3_FILE_NAME}"
GCS_URI = f"gs://{BUCKET_NAME}/{GCS_BLOB_NAME}"

print(f"   Destination: {GCS_URI}")
print(f"   Size: {S3_FILE_SIZE_MB:.2f} MB")
print()

# Record transfer start time for lineage
transfer_start_time = datetime.datetime.now(datetime.timezone.utc)
print(f"   Transfer started: {transfer_start_time.isoformat()}")

try:
    # Download from S3 to memory buffer
    print("   Downloading from S3...")
    buffer = io.BytesIO()
    s3_client.download_fileobj(S3_BUCKET, S3_FILE_KEY, buffer)
    buffer.seek(0)
    
    # Upload to GCS
    print("   Uploading to GCS...")
    blob = bucket.blob(GCS_BLOB_NAME)
    blob.upload_from_file(buffer, content_type='text/csv')
    
    # Record transfer end time
    transfer_end_time = datetime.datetime.now(datetime.timezone.utc)
    transfer_duration = (transfer_end_time - transfer_start_time).total_seconds()
    
    print()
    print(f"✅ Transfer completed successfully")
    print(f"   Duration: {transfer_duration:.2f} seconds")
    print(f"   Completed: {transfer_end_time.isoformat()}")
    print()
    print(f"   GCS URI: {GCS_URI}")
    
except Exception as e:
    print(f"❌ Transfer failed: {e}")
    transfer_end_time = datetime.datetime.now(datetime.timezone.utc)
    raise

📦 Transferring file from S3 to GCS...

   Source: s3://dataworld-linked-acs/TabulationQueries/pums_estimates_14.csv
   Destination: gs://johnswain-1200-20260303091357-census-s3-import/data/pums_estimates_14.csv
   Size: 0.09 MB

   Transfer started: 2026-03-10T08:39:24.604495+00:00
   Uploading to GCS...

✅ Transfer completed successfully
   Duration: 1.01 seconds
   Completed: 2026-03-10T08:39:25.616847+00:00

   GCS URI: gs://johnswain-1200-20260303091357-census-s3-import/data/pums_estimates_14.csv


# Section 6: Load Data into BigQuery

Create a BigQuery table and load the CSV data from GCS.

In [58]:
# Initialize BigQuery client
bq_client = bigquery.Client(project=PROJECT_ID)

print("✅ BigQuery client initialized")

✅ BigQuery client initialized


In [59]:
# Create or verify BigQuery dataset
print(f"📊 Creating/verifying BigQuery dataset: {BQ_DATASET_ID}")

dataset_id = f"{PROJECT_ID}.{BQ_DATASET_ID}"

try:
    dataset = bq_client.get_dataset(dataset_id)
    print(f"⚠️  Dataset '{BQ_DATASET_ID}' already exists")
    print(f"   Will use existing dataset")
except:
    dataset = bigquery.Dataset(dataset_id)
    dataset.location = "US"
    dataset = bq_client.create_dataset(dataset)
    print(f"✅ Dataset created: {dataset.dataset_id}")

📊 Creating/verifying BigQuery dataset: census_bureau_acs
⚠️  Dataset 'census_bureau_acs' already exists
   Will use existing dataset


In [60]:
# Load CSV from GCS to BigQuery
print(f"📥 Loading data into BigQuery table: {BQ_DATASET_ID}.{BQ_TABLE_ID}")
print()
print(f"   Source: {GCS_URI}")
print()

table_id = f"{PROJECT_ID}.{BQ_DATASET_ID}.{BQ_TABLE_ID}"

# Configure load job
job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,  # Skip header row
    autodetect=True,  # Auto-detect schema
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE  # Overwrite if exists
)

# Record load start time
load_start_time = datetime.datetime.now(datetime.timezone.utc)

try:
    # Start load job
    load_job = bq_client.load_table_from_uri(
        GCS_URI,
        table_id,
        job_config=job_config
    )
    
    print("   Loading... (this may take a few minutes)")
    load_job.result()  # Wait for job to complete
    
    load_end_time = datetime.datetime.now(datetime.timezone.utc)
    load_duration = (load_end_time - load_start_time).total_seconds()
    
    # Get table info
    table = bq_client.get_table(table_id)
    
    print()
    print(f"✅ Data loaded successfully")
    print(f"   Duration: {load_duration:.2f} seconds")
    print(f"   Rows: {table.num_rows:,}")
    print(f"   Columns: {len(table.schema)}")
    print()
    print(f"   BigQuery Table: {table_id}")
    print()
    print(f"🔗 View table in console:")
    print(f"   https://console.cloud.google.com/bigquery?project={PROJECT_ID}&p={PROJECT_ID}&d={BQ_DATASET_ID}&t={BQ_TABLE_ID}&page=table")
    
except Exception as e:
    print(f"❌ Load job failed: {e}")
    load_end_time = datetime.datetime.now(datetime.timezone.utc)
    raise

📥 Loading data into BigQuery table: census_bureau_acs.s3_acs_data

   Source: gs://johnswain-1200-20260303091357-census-s3-import/data/pums_estimates_14.csv

   Loading... (this may take a few minutes)

✅ Data loaded successfully
   Duration: 4.04 seconds
   Rows: 1,696
   Columns: 6

   BigQuery Table: johnswain-1200-20260303091357.census_bureau_acs.s3_acs_data

🔗 View table in console:
   https://console.cloud.google.com/bigquery?project=johnswain-1200-20260303091357&p=johnswain-1200-20260303091357&d=census_bureau_acs&t=s3_acs_data&page=table


In [61]:
# Preview data
print("📋 Data preview (first 5 rows):")
print()

query = f"""
SELECT *
FROM `{table_id}`
LIMIT 5
"""

try:
    df = bq_client.query(query).to_dataframe()
    print(df.to_string())
except Exception as e:
    print(f"⚠️  Could not preview data: {e}")

📋 Data preview (first 5 rows):



/Users/johnswain/Development/gcp-demo-dataplex/venv/lib/python3.11/site-packages/google/cloud/bigquery/table.py:1957: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


   st          state                            characteristic  pums_est_14  pums_se_14  pums_moe_14
0   0  United States                          Total population    318857056         459          754
1   0  United States       Housing unit population (RELP=0-15)    310792895         459          754
2   0  United States                GQ population (RELP=16-17)      8064161           0            0
3   0  United States     GQ institutional population (RELP=16)      3960839         984         1619
4   0  United States  GQ noninstitutional population (RELP=17)      4103322         984         1619


# Section 7: Create Dataplex Custom Entry for S3

Since Dataplex doesn't automatically catalog AWS S3 resources, we need to manually create a custom entry to represent the S3 source in the Universal Catalog.

In [62]:
# Initialize Dataplex Catalog client
catalog_client = dataplex_v1.CatalogServiceClient()

print("✅ Dataplex Catalog client initialized")

✅ Dataplex Catalog client initialized


In [63]:
# Step 7a: Create Custom Entry Group
print(f"📁 Creating Entry Group: {ENTRY_GROUP_ID}")
print()

entry_group_path = f"projects/{PROJECT_ID}/locations/{CATALOG_LOCATION}/entryGroups/{ENTRY_GROUP_ID}"

try:
    # Check if entry group exists
    try:
        existing_group = catalog_client.get_entry_group(name=entry_group_path)
        print(f"⚠️  Entry group '{ENTRY_GROUP_ID}' already exists")
        print(f"   Will use existing entry group")
    except:
        # Create new entry group
        entry_group = dataplex_v1.EntryGroup(
            description="External AWS S3 data sources for cross-cloud lineage tracking"
        )
        
        request = dataplex_v1.CreateEntryGroupRequest(
            parent=f"projects/{PROJECT_ID}/locations/{CATALOG_LOCATION}",
            entry_group_id=ENTRY_GROUP_ID,
            entry_group=entry_group
        )
        
        operation = catalog_client.create_entry_group(request=request)
        result = operation.result()
        
        print(f"✅ Entry group created: {result.name}")
    
    print()
    print(f"   Entry Group Path: {entry_group_path}")
    
except Exception as e:
    print(f"❌ Error creating entry group: {e}")
    raise

📁 Creating Entry Group: aws-storage-entries

⚠️  Entry group 'aws-storage-entries' already exists
   Will use existing entry group

   Entry Group Path: projects/johnswain-1200-20260303091357/locations/us/entryGroups/aws-storage-entries


In [64]:
# Step 7b: Create Custom Entry Type
print(f"📝 Creating Entry Type: {ENTRY_TYPE_ID}")
print()

entry_type_path = f"projects/{PROJECT_ID}/locations/{CATALOG_LOCATION}/entryTypes/{ENTRY_TYPE_ID}"

try:
    # Check if entry type exists
    try:
        existing_type = catalog_client.get_entry_type(name=entry_type_path)
        print(f"⚠️  Entry type '{ENTRY_TYPE_ID}' already exists")
        print(f"   Will use existing entry type")
    except:
        # Create new entry type
        entry_type = dataplex_v1.EntryType(
            description="AWS S3 object (bucket + key path)",
            platform="AWS S3",
            system="S3",
            type_aliases=["FILESET"]
        )
        
        request = dataplex_v1.CreateEntryTypeRequest(
            parent=f"projects/{PROJECT_ID}/locations/{CATALOG_LOCATION}",
            entry_type_id=ENTRY_TYPE_ID,
            entry_type=entry_type
        )
        
        operation = catalog_client.create_entry_type(request=request)
        result = operation.result()
        
        print(f"✅ Entry type created: {result.name}")
    
    print()
    print(f"   Entry Type Path: {entry_type_path}")
    
except Exception as e:
    print(f"❌ Error creating entry type: {e}")
    raise

📝 Creating Entry Type: s3-object

⚠️  Entry type 's3-object' already exists
   Will use existing entry type

   Entry Type Path: projects/johnswain-1200-20260303091357/locations/us/entryTypes/s3-object


In [65]:
# Step 7c: Create S3 Entry
print(f"📄 Creating S3 Entry for file: {S3_FILE_NAME}")
print()

# Create fully qualified name (FQN) for the S3 file
S3_FQN = f"s3://{S3_BUCKET}/{S3_FILE_KEY}"

# Sanitize entry ID (alphanumeric and hyphens only)
entry_id = re.sub(r'[^a-zA-Z0-9-]', '-', S3_FILE_NAME.replace('.csv', ''))
entry_id = f"s3-{entry_id}"

print(f"   Entry ID: {entry_id}")
print(f"   FQN: {S3_FQN}")
print()

entry_path = f"{entry_group_path}/entries/{entry_id}"

try:
    # Check if entry exists
    try:
        existing_entry = catalog_client.get_entry(name=entry_path)
        print(f"⚠️  Entry '{entry_id}' already exists")
        print(f"   Updating entry with current metadata")
        
        # Update existing entry
        existing_entry.fully_qualified_name = S3_FQN
        existing_entry.entry_source.description = f"ACS PUMS data from Data.world AWS S3 bucket. Original file: {S3_FILE_KEY}"
        existing_entry.entry_source.update_time = Timestamp(seconds=int(time.time()))
        
        update_mask = {
            "paths": ["fully_qualified_name", "entry_source.description", "entry_source.update_time"]
        }
        
        request = dataplex_v1.UpdateEntryRequest(
            entry=existing_entry,
            update_mask=update_mask
        )
        
        result = catalog_client.update_entry(request=request)
        print(f"✅ Entry updated: {result.name}")
        
    except:
        # Create new entry
        entry = dataplex_v1.Entry(
            name=entry_path,
            entry_type=entry_type_path,
            fully_qualified_name=S3_FQN,
            entry_source=dataplex_v1.EntrySource(
                resource=f"s3://{S3_BUCKET}/{S3_FILE_KEY}",
                system="AWS S3",
                platform="AWS",
                display_name=f"ACS PUMS Data - {S3_FILE_NAME}",
                description=f"American Community Survey (PUMS) data from Data.world. Original file: {S3_FILE_KEY}",
                create_time=Timestamp(seconds=int(time.time()))
            )
        )
        
        request = dataplex_v1.CreateEntryRequest(
            parent=entry_group_path,
            entry_id=entry_id,
            entry=entry
        )
        
        result = catalog_client.create_entry(request=request)
        print(f"✅ Entry created: {result.name}")
    
    print()
    print(f"   Entry Path: {entry_path}")
    print(f"   FQN: {S3_FQN}")
    
except Exception as e:
    print(f"❌ Error creating entry: {e}")
    print(f"   Details: {type(e).__name__}")
    raise

📄 Creating S3 Entry for file: pums_estimates_14.csv

   Entry ID: s3-pums-estimates-14
   FQN: s3://dataworld-linked-acs/TabulationQueries/pums_estimates_14.csv

⚠️  Entry 's3-pums-estimates-14' already exists
   Updating entry with current metadata
✅ Entry updated: projects/johnswain-1200-20260303091357/locations/us/entryGroups/aws-storage-entries/entries/s3-pums-estimates-14

   Entry Path: projects/johnswain-1200-20260303091357/locations/us/entryGroups/aws-storage-entries/entries/s3-pums-estimates-14
   FQN: s3://dataworld-linked-acs/TabulationQueries/pums_estimates_14.csv


# Section 8: Create Data Lineage Resources

Use the Data Lineage API to establish the connection between S3 and GCS in the Universal Catalog.

In [66]:
# Initialize Lineage client
lineage_client = LineageClient()

print("✅ Data Lineage client initialized")

✅ Data Lineage client initialized


In [67]:
# Step 8a: Create Process (the reusable transformation definition)
print(f"⚙️  Creating Lineage Process: {LINEAGE_PROCESS_ID}")
print()

process_parent = f"projects/{PROJECT_ID}/locations/{CATALOG_LOCATION}"
process_name = f"{process_parent}/processes/{LINEAGE_PROCESS_ID}"

try:
    # Check if process exists
    try:
        existing_process = lineage_client.get_process(name=process_name)
        print(f"⚠️  Process '{LINEAGE_PROCESS_ID}' already exists")
        print(f"   Will use existing process")
        process = existing_process
    except:
        # Create new process - name field sets the resource ID
        process = Process(
            name=process_name,
            display_name="S3 to GCS Data Import",
        )
        
        # Set attributes via the underlying protobuf to avoid proto-plus type mismatch
        process._pb.attributes["source_system"].CopyFrom(
            struct_pb2.Value(string_value="AWS S3")
        )
        process._pb.attributes["target_system"].CopyFrom(
            struct_pb2.Value(string_value="Google Cloud Storage")
        )
        process._pb.attributes["tool"].CopyFrom(
            struct_pb2.Value(string_value="Python boto3 + google-cloud-storage")
        )
        process._pb.attributes["transfer_type"].CopyFrom(
            struct_pb2.Value(string_value="cross-cloud")
        )
        
        request = CreateProcessRequest(
            parent=process_parent,
            process=process,
        )
        
        process = lineage_client.create_process(request=request)
        print(f"✅ Process created: {process.name}")
    
    print()
    print(f"   Process Name: {process.name}")
    print(f"   Display Name: {process.display_name}")
    
except Exception as e:
    print(f"❌ Error creating process: {e}")
    raise

⚙️  Creating Lineage Process: s3-to-gcs-import-process

✅ Process created: projects/212472948386/locations/us/processes/s3-to-gcs-import-process

   Process Name: projects/212472948386/locations/us/processes/s3-to-gcs-import-process
   Display Name: S3 to GCS Data Import


In [68]:
# Step 8b: Create Run (a specific execution of the process)
print(f"🏃 Creating Lineage Run for this execution")
print()

# Generate run ID with timestamp
run_id = f"run-{int(time.time())}"
run_name = f"{process.name}/runs/{run_id}"

try:
    # Create run timestamps
    start_timestamp = Timestamp()
    start_timestamp.FromDatetime(transfer_start_time)
    
    end_timestamp = Timestamp()
    end_timestamp.FromDatetime(transfer_end_time)
    
    run = Run(
        display_name=f"S3 to GCS Transfer - {S3_FILE_NAME}",
        state=Run.State.COMPLETED,
        start_time=start_timestamp,
        end_time=end_timestamp,
    )
    
    # Set attributes via the underlying protobuf
    run._pb.attributes["source_file"].CopyFrom(
        struct_pb2.Value(string_value=S3_FILE_KEY)
    )
    run._pb.attributes["destination_file"].CopyFrom(
        struct_pb2.Value(string_value=GCS_BLOB_NAME)
    )
    run._pb.attributes["file_size_mb"].CopyFrom(
        struct_pb2.Value(number_value=S3_FILE_SIZE_MB)
    )
    run._pb.attributes["transfer_duration_seconds"].CopyFrom(
        struct_pb2.Value(number_value=transfer_duration)
    )
    
    request = CreateRunRequest(
        parent=process.name,
        run=run,
    )
    
    run = lineage_client.create_run(request=request)
    
    print(f"✅ Run created: {run.name}")
    print()
    print(f"   Run ID: {run_id}")
    print(f"   State: {Run.State(run.state).name}")
    print(f"   Start Time: {transfer_start_time.isoformat()}")
    print(f"   End Time: {transfer_end_time.isoformat()}")
    
except Exception as e:
    print(f"❌ Error creating run: {e}")
    raise

🏃 Creating Lineage Run for this execution

✅ Run created: projects/212472948386/locations/us/processes/s3-to-gcs-import-process/runs/cacbd577-825a-4423-a62f-2bd6396b7d12

   Run ID: run-1773131988
   State: COMPLETED
   Start Time: 2026-03-10T08:39:24.604495+00:00
   End Time: 2026-03-10T08:39:25.616847+00:00


In [70]:
# Step 8c: Create Lineage Event (S3 → GCS)
from google.cloud.datacatalog_lineage_v1.types import CreateLineageEventRequest

print(f"🔗 Creating Lineage Event: S3 → GCS")
print()
print(f"   Source (S3):  {S3_FQN}")
print(f"   Target (GCS): {GCS_URI}")
print()

try:
    lineage_event = LineageEvent(
        start_time=start_timestamp,
        end_time=end_timestamp,
        links=[
            EventLink(
                source=EntityReference(fully_qualified_name=S3_FQN),
                target=EntityReference(fully_qualified_name=GCS_URI),
            )
        ],
    )

    request = CreateLineageEventRequest(
        parent=run.name,
        lineage_event=lineage_event,
    )

    s3_gcs_event = lineage_client.create_lineage_event(request=request)

    print(f"✅ Lineage event created: {s3_gcs_event.name}")
    print(f"   Links: {len(s3_gcs_event.links)}")

except Exception as e:
    print(f"❌ Error creating lineage event: {e}")
    raise

🔗 Creating Lineage Event: S3 → GCS

   Source (S3):  s3://dataworld-linked-acs/TabulationQueries/pums_estimates_14.csv
   Target (GCS): gs://johnswain-1200-20260303091357-census-s3-import/data/pums_estimates_14.csv

✅ Lineage event created: projects/212472948386/locations/us/processes/s3-to-gcs-import-process/runs/cacbd577-825a-4423-a62f-2bd6396b7d12/lineageEvents/61a40f32-7086-4019-894c-561fed46bb10
   Links: 1


# Section 9: Optional - Create GCS to BigQuery Lineage

Create a second lineage link showing the data journey from GCS to BigQuery.

In [71]:
# Create Process for GCS to BigQuery
print(f"⚙️  Creating Lineage Process for GCS to BigQuery")
print()

bq_process_id = "gcs-to-bigquery-load-process"
bq_process_name = f"{process_parent}/processes/{bq_process_id}"

try:
    # Check if process exists
    try:
        bq_process = lineage_client.get_process(name=bq_process_name)
        print(f"⚠️  Process '{bq_process_id}' already exists")
        print(f"   Will use existing process")
    except:
        # Create new process - name field sets the resource ID
        bq_process = Process(
            name=bq_process_name,
            display_name="GCS to BigQuery Load",
        )
        
        # Set attributes via the underlying protobuf
        bq_process._pb.attributes["source_system"].CopyFrom(
            struct_pb2.Value(string_value="Google Cloud Storage")
        )
        bq_process._pb.attributes["target_system"].CopyFrom(
            struct_pb2.Value(string_value="BigQuery")
        )
        bq_process._pb.attributes["tool"].CopyFrom(
            struct_pb2.Value(string_value="google-cloud-bigquery")
        )
        bq_process._pb.attributes["load_type"].CopyFrom(
            struct_pb2.Value(string_value="CSV")
        )
        
        request = CreateProcessRequest(
            parent=process_parent,
            process=bq_process,
        )
        
        bq_process = lineage_client.create_process(request=request)
        print(f"✅ Process created: {bq_process.name}")
    
    print()
    print(f"   Process Name: {bq_process.name}")
    
except Exception as e:
    print(f"❌ Error creating BigQuery process: {e}")
    raise

⚙️  Creating Lineage Process for GCS to BigQuery

✅ Process created: projects/212472948386/locations/us/processes/gcs-to-bigquery-load-process

   Process Name: projects/212472948386/locations/us/processes/gcs-to-bigquery-load-process


In [72]:
# Create Run for BigQuery load
print(f"🏃 Creating Run for BigQuery load")
print()

bq_run_id = f"run-bq-{int(time.time())}"

try:
    # Create run timestamps
    bq_start_timestamp = Timestamp()
    bq_start_timestamp.FromDatetime(load_start_time)
    
    bq_end_timestamp = Timestamp()
    bq_end_timestamp.FromDatetime(load_end_time)
    
    bq_run = Run(
        display_name=f"Load {S3_FILE_NAME} to BigQuery",
        state=Run.State.COMPLETED,
        start_time=bq_start_timestamp,
        end_time=bq_end_timestamp,
    )
    
    # Set attributes via the underlying protobuf
    bq_run._pb.attributes["source_uri"].CopyFrom(
        struct_pb2.Value(string_value=GCS_URI)
    )
    bq_run._pb.attributes["destination_table"].CopyFrom(
        struct_pb2.Value(string_value=table_id)
    )
    bq_run._pb.attributes["rows_loaded"].CopyFrom(
        struct_pb2.Value(number_value=float(table.num_rows))
    )
    bq_run._pb.attributes["load_duration_seconds"].CopyFrom(
        struct_pb2.Value(number_value=load_duration)
    )
    
    request = CreateRunRequest(
        parent=bq_process.name,
        run=bq_run,
    )
    
    bq_run = lineage_client.create_run(request=request)
    
    print(f"✅ Run created: {bq_run.name}")
    print()
    print(f"   Run ID: {bq_run_id}")
    print(f"   Rows Loaded: {table.num_rows:,}")
    
except Exception as e:
    print(f"❌ Error creating BigQuery run: {e}")
    raise

🏃 Creating Run for BigQuery load

✅ Run created: projects/212472948386/locations/us/processes/gcs-to-bigquery-load-process/runs/b37c0d2f-27fb-42be-a1b2-57dc4e150165

   Run ID: run-bq-1773132164
   Rows Loaded: 1,696


In [73]:
# Create Lineage Event (GCS → BigQuery)
BQ_FQN = f"bigquery:{table_id}"

print(f"🔗 Creating Lineage Event: GCS → BigQuery")
print()
print(f"   Source (GCS):      {GCS_URI}")
print(f"   Target (BigQuery): {BQ_FQN}")
print()

try:
    bq_lineage_event = LineageEvent(
        start_time=bq_start_timestamp,
        end_time=bq_end_timestamp,
        links=[
            EventLink(
                source=EntityReference(fully_qualified_name=GCS_URI),
                target=EntityReference(fully_qualified_name=BQ_FQN),
            )
        ],
    )

    request = CreateLineageEventRequest(
        parent=bq_run.name,
        lineage_event=bq_lineage_event,
    )

    gcs_bq_event = lineage_client.create_lineage_event(request=request)

    print(f"✅ Lineage event created: {gcs_bq_event.name}")
    print(f"   Links: {len(gcs_bq_event.links)}")

except Exception as e:
    print(f"❌ Error creating BigQuery lineage event: {e}")
    raise

🔗 Creating Lineage Event: GCS → BigQuery

   Source (GCS):      gs://johnswain-1200-20260303091357-census-s3-import/data/pums_estimates_14.csv
   Target (BigQuery): bigquery:johnswain-1200-20260303091357.census_bureau_acs.s3_acs_data

✅ Lineage event created: projects/212472948386/locations/us/processes/gcs-to-bigquery-load-process/runs/b37c0d2f-27fb-42be-a1b2-57dc4e150165/lineageEvents/321f19db-3c5b-4655-9896-89668e76b94e
   Links: 1


# Section 10: Validation and Console Links

Verify all resources were created successfully and provide links to view in the GCP Console.

In [74]:
# Validation Summary
print("=" * 70)
print("📊 SETUP COMPLETE - VALIDATION SUMMARY")
print("=" * 70)
print()

print("✅ RESOURCES CREATED:")
print()

print("1️⃣  GCS Bucket:")
print(f"   Name: {BUCKET_NAME}")
print(f"   File: {GCS_BLOB_NAME}")
print(f"   URI: {GCS_URI}")
print()

print("2️⃣  BigQuery Table:")
print(f"   Table: {table_id}")
print(f"   Rows: {table.num_rows:,}")
print(f"   Columns: {len(table.schema)}")
print()

print("3️⃣  Dataplex Entry Group:")
print(f"   ID: {ENTRY_GROUP_ID}")
print(f"   Path: {entry_group_path}")
print()

print("4️⃣  Dataplex Entry Type:")
print(f"   ID: {ENTRY_TYPE_ID}")
print(f"   Path: {entry_type_path}")
print()

print("5️⃣  S3 Custom Entry:")
print(f"   Entry ID: {entry_id}")
print(f"   FQN: {S3_FQN}")
print(f"   Path: {entry_path}")
print()

print("6️⃣  Data Lineage Processes:")
print(f"   S3 to GCS: {process.name}")
print(f"   GCS to BigQuery: {bq_process.name}")
print()

print("7️⃣  Data Lineage Runs:")
print(f"   S3 to GCS Run: {run.name}")
print(f"   GCS to BigQuery Run: {bq_run.name}")
print()

print("8️⃣  Lineage Events:")
print(f"   S3 → GCS: Created")
print(f"   GCS → BigQuery: Created")
print()

print("=" * 70)

📊 SETUP COMPLETE - VALIDATION SUMMARY

✅ RESOURCES CREATED:

1️⃣  GCS Bucket:
   Name: johnswain-1200-20260303091357-census-s3-import
   File: data/pums_estimates_14.csv
   URI: gs://johnswain-1200-20260303091357-census-s3-import/data/pums_estimates_14.csv

2️⃣  BigQuery Table:
   Table: johnswain-1200-20260303091357.census_bureau_acs.s3_acs_data
   Rows: 1,696
   Columns: 6

3️⃣  Dataplex Entry Group:
   ID: aws-storage-entries
   Path: projects/johnswain-1200-20260303091357/locations/us/entryGroups/aws-storage-entries

4️⃣  Dataplex Entry Type:
   ID: s3-object
   Path: projects/johnswain-1200-20260303091357/locations/us/entryTypes/s3-object

5️⃣  S3 Custom Entry:
   Entry ID: s3-pums-estimates-14
   FQN: s3://dataworld-linked-acs/TabulationQueries/pums_estimates_14.csv
   Path: projects/johnswain-1200-20260303091357/locations/us/entryGroups/aws-storage-entries/entries/s3-pums-estimates-14

6️⃣  Data Lineage Processes:
   S3 to GCS: projects/212472948386/locations/us/processes/s3-to-

In [75]:
# Console Links
print()
print("=" * 70)
print("🔗 VIEW RESOURCES IN GCP CONSOLE")
print("=" * 70)
print()

print("📦 GCS Bucket:")
print(f"https://console.cloud.google.com/storage/browser/{BUCKET_NAME}?project={PROJECT_ID}")
print()

print("📊 BigQuery Table:")
print(f"https://console.cloud.google.com/bigquery?project={PROJECT_ID}&ws=!1m5!1m4!4m3!1s{PROJECT_ID}!2s{BQ_DATASET_ID}!3s{BQ_TABLE_ID}")
print()

print("🗂️  Dataplex Entry Groups:")
print(f"https://console.cloud.google.com/dataplex/govern/entry-groups?project={PROJECT_ID}")
print()

print("🔍 Dataplex Search (search for your data):")
print(f"https://console.cloud.google.com/dataplex/search?project={PROJECT_ID}")
print()

print("🔗 Data Lineage (search for the BigQuery table to see lineage):")
print(f"   1. Go to Dataplex Search: https://console.cloud.google.com/dataplex/search?project={PROJECT_ID}")
print(f"   2. Search for: {BQ_TABLE_ID}")
print(f"   3. Click on the table result")
print(f"   4. Navigate to the 'Lineage' tab")
print(f"   5. You should see: S3 → GCS → BigQuery")
print()

print("=" * 70)


🔗 VIEW RESOURCES IN GCP CONSOLE

📦 GCS Bucket:
https://console.cloud.google.com/storage/browser/johnswain-1200-20260303091357-census-s3-import?project=johnswain-1200-20260303091357

📊 BigQuery Table:
https://console.cloud.google.com/bigquery?project=johnswain-1200-20260303091357&ws=!1m5!1m4!4m3!1sjohnswain-1200-20260303091357!2scensus_bureau_acs!3ss3_acs_data

🗂️  Dataplex Entry Groups:
https://console.cloud.google.com/dataplex/govern/entry-groups?project=johnswain-1200-20260303091357

🔍 Dataplex Search (search for your data):
https://console.cloud.google.com/dataplex/search?project=johnswain-1200-20260303091357

🔗 Data Lineage (search for the BigQuery table to see lineage):
   1. Go to Dataplex Search: https://console.cloud.google.com/dataplex/search?project=johnswain-1200-20260303091357
   2. Search for: s3_acs_data
   3. Click on the table result
   4. Navigate to the 'Lineage' tab
   5. You should see: S3 → GCS → BigQuery



# How to View the Lineage Graph

## In the Dataplex Console:

1. **Navigate to Dataplex Search**:
   - Go to the [Dataplex Search Console](https://console.cloud.google.com/dataplex/search)
   - Or navigate: Cloud Console → Dataplex → Search

2. **Search for the BigQuery table**:
   - Enter the table name in the search box (e.g., `s3_acs_data`)
   - Click on the table result

3. **View the Lineage tab**:
   - Click on the **Lineage** tab in the entry details
   - You should see a visual graph showing:
     - **S3 Entry** (left) → your AWS S3 source
     - **GCS File** (middle) → the Cloud Storage copy
     - **BigQuery Table** (right) → the final table

4. **Explore the graph**:
   - Click on nodes to see details
   - Hover over edges to see process and run information
   - Use the timeline to see when data moved

## What the Lineage Shows

The lineage graph demonstrates:

- **Data Origin**: The S3 custom entry shows data came from AWS
- **Transformation Path**: Two processes show S3→GCS and GCS→BigQuery
- **Time Stamps**: When each transfer occurred
- **Metadata**: File sizes, row counts, and transfer durations

This enables:
- **Impact Analysis**: See what would be affected by changes to the S3 source
- **Compliance**: Document data provenance for audits
- **Discovery**: Find data by its original source (AWS S3)
- **Understanding**: Trace data journey across cloud platforms

# Next Steps

## Cleanup

When you're done exploring the lineage graph, run the cleanup notebook to remove all resources:

```bash
jupyter notebook cleanup/07_cleanup_s3_lineage.ipynb
```

This will delete:
- GCS bucket and files
- BigQuery table
- Custom Dataplex entries, entry group, and entry type
- Data Lineage processes, runs, and events

**Important**: The CloudSQL instance and other resources from previous notebooks are NOT affected by this cleanup.

## Further Exploration

1. **Add More S3 Files**: Modify the notebook to transfer multiple files
2. **Apply Aspects**: Add the governance aspects to the S3 entry
3. **Create Data Quality Rules**: Define DQ rules for the BigQuery table
4. **Automate**: Convert this notebook into a scheduled Cloud Function
5. **Multi-Cloud Strategy**: Expand to include Azure Blob Storage or other sources